# 01 - Fixed-size chunking

Fixed-size chunking splits text into chunks with the same target number of words. It does not understand sentences, paragraphs, headings, or legal structure. That is exactly why it is useful as a baseline: every later method should prove that it improves something over this simple approach.

In RAG, fixed-size chunking answers this question: *What happens if every chunk has a predictable size?*

## What this notebook teaches
* How to extract text from a sectioned text document.
* How to split each section into fixed-size word windows.
* Why fixed-size chunks are easy to compare but can cut evidence in the wrong place.
* What metadata each chunk should carry for RAG citations.

In [ ]:
from pathlib import Path
from pprint import pprint
import numpy as np

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data" / "agenticed_content.txt").is_file()),
     None
)

file_path = ROOT / "data" / "agenticed_content.txt"
content = file_path.read_text(encoding="utf-8")
lines = content.splitlines()

## Grouping the Document by Sections
Before chunking, we need to parse the text document into its logical sections. This prevents our chunks from accidentally merging the end of one section with the beginning of the next.

In [ ]:
sections = []
current_heading = None
current_lines = []

for line in lines:
    line = line.strip()

    if line.startswith("SECTION"):
        if current_heading is not None:
            sections.append({
                "section": current_heading,
                "text": "\n".join(current_lines).strip()
            })
        current_heading = line
        current_lines = []

    else:
        current_lines.append(line)

if current_heading is not None:
    sections.append({
        "section": current_heading,
        "text": "\n".join(current_lines).strip()
    })

In [ ]:
print("Total sections: ", len(sections))

for section in sections:
    print(section["section"], "|", len(section["text"].split()), "words")

## Fixed-size chunking function

This function works section by section. For each section, it splits the text into words and creates a new chunk every `words_per_chunk` words.

Important behavior:
* `start_word` and `end_word` are word positions, not character positions.
* The last chunk in a section can be smaller than `words_per_chunk`.
* A sentence can be cut in half because the function only counts words.
* Section metadata is preserved so retrieved chunks can be cited accurately.

In [ ]:
def fixed_size_chunking(sections, words_per_chunk=180):
    chunks = []

    for section in sections:
        words = section["text"].split()

        for start in range(0, len(words), words_per_chunk):
            end = min(start + words_per_chunk, len(words))

            chunk_words = words[start:end]

            chunks.append({
                "chunk-id": f"fixed-{len(chunks) + 1}",
                "source": file_path.name,
                "section": section["section"],
                "start_word": start + 1,
                "end_word": end,
                "text": " ".join(chunk_words)
            })

    return chunks

In [ ]:
fixed_size_chunks = fixed_size_chunking(sections)

In [ ]:
print(f"Total fixed-size chunks: {len(fixed_size_chunks)}")
type(fixed_size_chunks)
pprint(fixed_size_chunks[0])
print("\n\n\n")
pprint(fixed_size_chunks[1])

## Inspecting Section-Based Fixed-Size Chunking

Because the fixed-size function works section by section with a limit of 180 words, sections shorter than the limit fit in one chunk. Sections 1, 2, 3, 5, and 6 do this. Section 4 (the FAQ, 428 words) is longer, so the function splits it into three windows. Below, we inspect the chunk metadata and the split points.

In [ ]:
# Show any fixed-size split that falls inside a section (none in this file).
for a, b in zip(fixed_size_chunks, fixed_size_chunks[1:]):
    if a["section"] == b["section"]:
        print(f"Split inside {a['section']}: {a['chunk-id']} -> {b['chunk-id']}")

print("\nTotal fixed-size chunks:", len(fixed_size_chunks))

## Chunk Statistics and Verification

In [ ]:
word_counts = [len(chunk["text"].split()) for chunk in fixed_size_chunks]

min_words = min(word_counts)
max_words = max(word_counts)
avg_words = np.array(word_counts).mean()

print(f"Minimum words:  {min_words}")
print(f"Maximum words:  {max_words}")
print(f"Average words:  {avg_words:.2f}")

In [ ]:
for idx in [0, 1, len(fixed_size_chunks)-1]:
    chunk = fixed_size_chunks[idx]
    
    print(f"--- Chunk ID: {chunk['chunk-id']} ---")
    print(f"Section:    {chunk['section']}")
    print(f"Word Span:  {chunk['start_word']} to {chunk['end_word']}")
    print(f"Word Count: {len(chunk['text'].split())}")
    print(f"Snippet:    {chunk['text'][:90]}...\n")

## Summary and Observations: Fixed-Size Chunking

### 1. Key Metrics
- **Total Chunks Generated:** 8
- **Maximum Chunk Size:** 180 words
- **Minimum Chunk Size:** 26 words
- **Average Chunk Size:** 101.88 words
- **Source File:** `agenticed_content.txt` (across 6 sections)

### 2. Advantages of Fixed-Size Chunking
- **Predictable Size:** Chunks stay strictly within a predefined word limit, preventing context window overflow.
- **Implementation Simplicity:** Extremely easy and computationally inexpensive to implement using basic word slicing.
- **Section Isolation:** By parsing text section-by-section before chunking, chunks never bleed across completely unrelated headings.

### 3. Limitations and Impact on RAG
- **Rigid Boundaries:** Fixed word windows count tokens/words blindly without respecting sentence or paragraph boundaries.
- **Context Fragmentation:** Section 4 shows the problem. Chunk 4 ends with "with your discussion", and chunk 5 starts with "depth.", so a sentence is cut in half and the retrieval engine gets incomplete context.
- **Varying Chunk Densities:** Shorter sections produce smaller standalone chunks (for example, 26 words for Section 6), which may lack sufficient surrounding context for effective semantic matching.

# 02 - Sentence-Based Chunking

Sentence-based chunking splits text using natural grammatical boundaries (such as periods, question marks, and exclamation points) rather than an arbitrary word count. This prevents sentences from being cut in half, preserving semantic integrity for RAG.

## What this section teaches
* How to tokenize text into sentences.
* How to group sentences together up to a target word/token limit.
* Why sentence-based chunking improves retrieval quality over fixed-size chunking.

In [ ]:
import re

def split_sentence(text):

    return [part.strip() for part in re.split(r"(?<=[.!?])\s+", text) if part.strip()]

In [ ]:
sample_sentences = split_sentence(sections[0]["text"])

In [ ]:
print(f"Total sentences found in Section 1: {len(sample_sentences)}")
for idx, sent in enumerate(sample_sentences):
    print(f"\n[Sentence {idx+1}]: {sent}")

In [ ]:
def sentence_based_chunking(sections, max_words_per_chunk=180):
    chunks = []

    for section in sections:

        sentences = split_sentence(section["text"])

        current_chunk_sentences = []
        current_word_count = 0

        for sentence in sentences:
            sentence_words = sentence.split()
            sentence_length = len(sentence_words)

            if current_chunk_sentences and (current_word_count + sentence_length > max_words_per_chunk):

                chunks.append({
                    "chunk_id": f"sentence-{len(chunks) + 1}",
                    "source": file_path.name,
                    "section": section["section"],
                    "text": " ".join(current_chunk_sentences)
                })

                current_chunk_sentences = []
                current_word_count = 0


            current_chunk_sentences.append(sentence)
            current_word_count += sentence_length


        if current_chunk_sentences:
            chunks.append({
                "chunk_id": f"sentence-{len(chunks) + 1}",
                "source": file_path.name,
                "section": section["section"],
                "text": " ".join(current_chunk_sentences)
            })

    return chunks
        

In [ ]:
sentence_chunks = sentence_based_chunking(sections)
print(f"Total sentence based chunks: {len(sentence_chunks)}")

In [ ]:
type(sentence_chunks)

In [ ]:
pprint(sentence_chunks[0], width=50)

In [ ]:
chunk_word_counts = [len(c["text"].split()) for c in sentence_chunks]

for i, count in enumerate(chunk_word_counts):
    print(f"Chunk {i}: {count} words")

In [ ]:
total_chunks = len(sentence_chunks)
min_words = min(chunk_word_counts)
max_words = max(chunk_word_counts)
avg_words = np.array(chunk_word_counts).mean()

print(f"Total Chunks:   {total_chunks}")
print(f"Min Word Count: {min_words}")
print(f"Max Word Count: {max_words}")
print(f"Avg Word Count: {avg_words:.1f}")

In [ ]:
print("End of chunk 1: \n")
print(sentence_chunks[0]["text"][-120:])

print("\nBeginning of chunk 2:")
print(sentence_chunks[1]["text"][:120])

In [ ]:
print("Inspecting Chunk 2 metadata and structure:")
pprint(sentence_chunks[1], width=70)

### Advantages and Limitations of Sentence-Based Chunking

#### Advantages:
1. **Preserves Complete Thoughts**: Sentences are never cut in half mid-phrase, ensuring every retrieved chunk makes grammatical sense to an LLM.
2. **Context Window Control**: Uses a soft word budget (180 words) to prevent chunks from growing excessively large while respecting sentence boundaries.
3. **Structured Attribution**: Every chunk retains its section heading and source file name for accurate RAG citations.

#### Limitations:
1. **Variable Chunk Sizes**: Because sentences vary in length, chunks are not uniform in size. In this file, sentence-based chunks range from 26 to 179 words. Section 4 becomes three chunks of 171, 179, and 78 words.
2. **Regex Splitting Sensitivity**: Abbreviations (e.g., "Dr.", "e.g.", "vs.") or complex punctuation can occasionally trigger false sentence splits if not handled by advanced tokenizers.

# 03 - Paragraph-Based Chunking

Paragraph-based chunking splits text using natural structural boundaries (such as blank lines between paragraphs) rather than an arbitrary word count. This keeps related sentences grouped together into coherent topical units, preserving contextual integrity for RAG.

## What this section teaches

* How to identify and split text across paragraph boundaries.
* How to group paragraphs together up to a target word limit while preserving structural units.
* Why paragraph-based chunking improves topical context and coherence over fixed-size and sentence-based chunking.

In [ ]:
def split_paragraph(text):

    raw_paras = text.split("\n\n")

    return [p.strip() for p in raw_paras if p.strip()]

In [ ]:
sample_paragraphs = split_paragraph(sections[0]["text"])

print(f"Total paragraphs found in Section 1: {len(sample_paragraphs)}")

In [ ]:
for i, section in enumerate(sections):
    paras = split_paragraph(section["text"])
    print(f"{section['section']}: {len(paras)} paragraph(s)")

In [ ]:
def paragraph_based_chunking(sections, max_words_per_chunk=180):
    chunks = []

    for section in sections:

        paragraphs = split_paragraph(section["text"])

        current_chunk_paragraphs = []
        current_word_counts = 0

        for paragraph in paragraphs:

            para_words = paragraph.split()
            para_length = len(para_words)

            if current_chunk_paragraphs and (current_word_counts + para_length > max_words_per_chunk):
                chunks.append({
                    "chunk_id": f"paragraph-{len(chunks) + 1}",
                    "source": file_path.name,
                    "section": section["section"],
                    "text": "\n\n".join(current_chunk_paragraphs)
                })

                current_chunk_paragraphs = []
                current_word_counts = 0


            current_chunk_paragraphs.append(paragraph)
            current_word_counts += para_length


        if current_chunk_paragraphs:
            chunks.append({
                "chunk_id": f"paragraph-{len(chunks) + 1}",
                "source": file_path.name,
                "section": section["section"],
                "text": "\n\n".join(current_chunk_paragraphs)
            })

    return chunks

In [ ]:
paragraph_chunks = paragraph_based_chunking(sections)

print(f"Total paragraph-based chunks: {len(paragraph_chunks)}")

In [ ]:
para_word_counts = [len(c["text"].split()) for c in paragraph_chunks]

for i, count in enumerate(para_word_counts):
    print(f"Paragraph Chunk {i+1}: {count} words")

In [ ]:
total_para_chunks = len(paragraph_chunks)
min_para_words = min(para_word_counts)
max_para_words = max(para_word_counts)
avg_para_words = np.array(para_word_counts).mean()

print(f"Total Paragraph Chunks: {total_para_chunks}")
print(f"Min Word Count:         {min_para_words}")
print(f"Max Word Count:         {max_para_words}")
print(f"Avg Word Count:         {avg_para_words:.1f}")

In [ ]:
print("Inspecting Chunk 2 structure and metadata:")
pprint(paragraph_chunks[1])

### Advantages and Limitations of Paragraph-Based Chunking

#### Advantages:
1. **Preserves Thematic Context**: Paragraphs are author-defined units of thought; keeping them intact preserves complete conceptual ideas and context for retrieval.
2. **Maintains Document Layout**: Uses `

` to preserve structured formatting (such as subheadings, bullet lists, or section blocks) within a single chunk.
3. **Structured Attribution**: Retains exact section metadata and source filenames for citation in downstream LLM responses.

#### Limitations:
1. **Susceptible to Outliers (Budget Overruns)**: If a source text contains an unusually long paragraph without blank lines, it cannot be broken down without violating paragraph integrity. Every paragraph in this file is under 180 words, so no paragraph is split.
2. **High Word-Count Variance**: Chunk lengths range from 26 to 152 words in this file, which can make LLM prompt window budgeting less predictable.

# 04 - Comparison of Chunking Strategies

### Summary Metrics Across All Three Methods

| Strategy | Total Chunks | Min Words | Max Words | Avg Words | Sections Split |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Fixed-Size Chunking** | 8 | 26 | 180 | 101.9 | Section 4, mid-sentence |
| **Sentence-Based Chunking** | 8 | 26 | 179 | 101.9 | Section 4, between sentences |
| **Paragraph-Based Chunking** | 8 | 26 | 152 | 101.9 | Section 4, between paragraphs |

---

### Key Observations from the Experiments

1. **Fixed-size chunking cuts sentences.** Section 4 is split at exactly 180 words, which falls mid-sentence twice: "...with your discussion" / "depth." and "...they've worked" / "through the weeks...".
2. **Sentence-based chunking keeps sentences whole.** Every break falls between sentences, and every chunk stays under the 180-word budget.
3. **Paragraph-based chunking keeps paragraphs whole.** Section 4 is three chunks of 130, 152, and 146 words, one or more whole Q&A pairs each.

4. **Which method is best for an AgenticEd Chatbot?**
   * **Sentence-based chunking with a soft word budget** is the best overall choice for this file. It never cuts a sentence and keeps chunks under the budget.
   * **Paragraph-based chunking** keeps each question and answer together, which suits FAQ-style content.